# EDA — HAM10000 & Fitzpatrick17k
Owner: Afzal Raza. Run from the repo root after downloading data (see `data/README.md`). Outputs are cleared before commit; figures are saved to `reports/figures/`.

In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from skinfair import CLASSES
from skinfair.data.datasets import load_ham_metadata
from skinfair.data.fitzpatrick_mapping import mapping_report, map_label
from skinfair.data.splits import lesion_level_split, assert_no_leakage

# notebooks are run from notebooks/; go to the repo root so data/ and reports/ paths work
if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)

HAM, FITZ = Path('data/HAM10000'), Path('data/fitzpatrick17k')
FIG = Path('reports/figures'); FIG.mkdir(parents=True, exist_ok=True)

## 1. HAM10000 class distribution

In [ ]:
meta = load_ham_metadata(HAM)
counts = meta['dx'].value_counts().reindex(CLASSES)
print(counts)
print('imbalance ratio (max/min):', counts.max() / counts.min())
print('images:', len(meta), ' unique lesions:', meta['lesion_id'].nunique())
ax = counts.plot.bar(title='HAM10000 class distribution'); ax.set_ylabel('images')
plt.tight_layout(); plt.savefig(FIG / 'eda_ham_classes.png', dpi=150)

## 2. Lesion-level split check (no leakage) and per-split class balance

In [ ]:
tr, va, te = lesion_level_split(meta, seed=42)
assert_no_leakage(tr, va, te)
split_counts = pd.DataFrame({'train': tr['dx'].value_counts(), 'val': va['dx'].value_counts(), 'test': te['dx'].value_counts()}).reindex(CLASSES)
display(split_counts); display((split_counts / split_counts.sum()).round(3))
print('multi-image lesions:', (meta.groupby('lesion_id').size() > 1).sum())

## 3. Fitzpatrick17k: skin-type distribution and label mapping
Only a subset of its 114 conditions maps onto HAM classes; this is the domain-shift caveat for the report.

In [ ]:
fz = pd.read_csv(FITZ / 'fitzpatrick17k.csv')
print('total rows:', len(fz), ' conditions:', fz['label'].nunique())
rep = mapping_report(fz)
print('kept', rep['kept'], 'dropped', rep['dropped'], dict(rep['per_class']))
print('mapping keys NOT found in CSV (fix typos):', rep['missing_keys'])
print('most common unmapped labels:', rep['unmapped'].most_common(15))

In [ ]:
fz['dx'] = fz['label'].map(map_label)
mapped = fz[fz['dx'].notna()]
fst = fz['fitzpatrick_scale'].value_counts().sort_index()
fst_m = mapped['fitzpatrick_scale'].value_counts().sort_index()
pd.DataFrame({'all': fst, 'mapped': fst_m}).plot.bar(title='Fitzpatrick scale (-1 = unknown)')
plt.tight_layout(); plt.savefig(FIG / 'eda_fitz_fst.png', dpi=150)
print(pd.crosstab(mapped['dx'], mapped['fitzpatrick_scale']))  # small cells => wide CIs in the fairness analysis

## 4. Sample grids

In [ ]:
fig, axes = plt.subplots(len(CLASSES), 5, figsize=(10, 2 * len(CLASSES)))
for i, c in enumerate(CLASSES):
    for j, iid in enumerate(meta[meta['dx'] == c].sample(5, random_state=0)['image_id']):
        axes[i, j].imshow(Image.open(HAM / 'images' / f'{iid}.jpg')); axes[i, j].axis('off')
    axes[i, 0].set_title(c, loc='left', fontsize=9)
plt.tight_layout(); plt.savefig(FIG / 'eda_ham_samples.png', dpi=120)

In [ ]:
have = mapped[mapped['md5hash'].map(lambda h: (FITZ / 'images' / f'{h}.jpg').exists())]
fig, axes = plt.subplots(6, 5, figsize=(10, 12))
for i, s in enumerate(range(1, 7)):
    sub = have[have['fitzpatrick_scale'] == s]
    for j, (_, r) in enumerate(sub.sample(min(5, len(sub)), random_state=0).iterrows()):
        axes[i, j].imshow(Image.open(FITZ / 'images' / f'{r.md5hash}.jpg')); axes[i, j].axis('off')
    axes[i, 0].set_title(f'FST {s}', loc='left', fontsize=9)
plt.tight_layout(); plt.savefig(FIG / 'eda_fitz_samples.png', dpi=120)